# SHAP Sample Size Validation (R1-1)
Uses existing bootstrap arrays to validate ranking stability at different subset sizes.

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

results_dir = Path('../results')
tables_dir = results_dir / 'tables'

# Load bootstrap feature importance (1000 bootstrap iterations × 6 features)
boot_feat = np.load(results_dir / 'bootstrap_feature_importance.npy')
# Load bootstrap temporal importance (1000 bootstrap iterations × 15 timesteps)
boot_temp = np.load(results_dir / 'bootstrap_temporal_importance.npy')

print(f"Bootstrap feature importance shape: {boot_feat.shape}")
print(f"Bootstrap temporal importance shape: {boot_temp.shape}")

features = ['humidity', 'temperature', 'humiditysol', 'temperaturesol', 'co2', 'lumière']
n_boot = boot_feat.shape[0]

## ANALYSIS 1: Feature importance ranking stability

In [ ]:
print("\n" + "="*60)
print("FEATURE IMPORTANCE RANKING STABILITY")
print("="*60)

# Full bootstrap (all 1000 iterations)
full_mean = np.mean(boot_feat, axis=0)
full_rank = np.argsort(full_mean)[::-1]
print(f"\nFull bootstrap ranking: {[features[i] for i in full_rank]}")
print(f"Full bootstrap importance: {full_mean[full_rank]}")

# Test stability with subsets of bootstrap iterations
for subset_frac in [0.25, 0.5, 0.75, 1.0]:
    n_sub = int(n_boot * subset_frac)
    ranking_matches = 0
    top3_matches = 0
    n_trials = 500

    for trial in range(n_trials):
        idx = np.random.choice(n_boot, n_sub, replace=False)
        sub_mean = np.mean(boot_feat[idx], axis=0)
        sub_rank = np.argsort(sub_mean)[::-1]

        if sub_rank[0] == full_rank[0]:
            ranking_matches += 1
        if set(sub_rank[:3]) == set(full_rank[:3]):
            top3_matches += 1

    print(f"  {n_sub} iterations: Top-1 stable={ranking_matches/n_trials*100:.1f}%, Top-3 stable={top3_matches/n_trials*100:.1f}%")

## ANALYSIS 2: CI width convergence

In [ ]:
print("\n" + "="*60)
print("CI WIDTH CONVERGENCE ANALYSIS")
print("="*60)

ci_results = []
for n_iter in [100, 250, 500, 750, 1000]:
    n_sub = min(n_iter, n_boot)

    # Run 50 random draws of n_sub bootstrap iterations
    ci_widths_per_trial = []
    for trial in range(50):
        idx = np.random.choice(n_boot, n_sub, replace=False)
        sub = boot_feat[idx]
        ci_lower = np.percentile(sub, 2.5, axis=0)
        ci_upper = np.percentile(sub, 97.5, axis=0)
        ci_width = ci_upper - ci_lower
        ci_widths_per_trial.append(ci_width)

    mean_ci_width = np.mean(ci_widths_per_trial, axis=0)
    # Relative CI = CI width / mean importance
    mean_importance = np.mean(boot_feat[:n_sub], axis=0)
    rel_ci = mean_ci_width / (mean_importance + 1e-10) * 100

    # Only report for features with non-zero importance
    active_features = mean_importance > 1e-6

    result = {
        'Bootstrap_Iterations': n_sub,
        'Mean_CI_Width': float(np.mean(mean_ci_width[active_features])),
        'Mean_Relative_CI_Pct': float(np.mean(rel_ci[active_features])),
        'Max_Relative_CI_Pct': float(np.max(rel_ci[active_features])),
    }
    ci_results.append(result)
    print(f"  n={n_sub}: Mean CI width={result['Mean_CI_Width']:.6f}, "
          f"Mean Rel CI={result['Mean_Relative_CI_Pct']:.1f}%, "
          f"Max Rel CI={result['Max_Relative_CI_Pct']:.1f}%")

## ANALYSIS 3: Per-feature CI analysis

In [ ]:
print("\n" + "="*60)
print("PER-FEATURE CONFIDENCE INTERVAL ANALYSIS")
print("="*60)

feat_ci = []
for i, feat in enumerate(features):
    vals = boot_feat[:, i]
    mean_val = np.mean(vals)
    ci_lower = np.percentile(vals, 2.5)
    ci_upper = np.percentile(vals, 97.5)
    ci_width = ci_upper - ci_lower
    rel_ci = ci_width / (mean_val + 1e-10) * 100 if mean_val > 1e-6 else float('nan')

    feat_ci.append({
        'Feature': feat,
        'Mean_Importance': float(mean_val),
        'CI_Lower': float(ci_lower),
        'CI_Upper': float(ci_upper),
        'CI_Width': float(ci_width),
        'Relative_CI_Pct': float(rel_ci) if not np.isnan(rel_ci) else 'N/A'
    })
    print(f"  {feat:15s}: {mean_val:.6f} [{ci_lower:.6f}, {ci_upper:.6f}] "
          f"(width={ci_width:.6f}, rel={rel_ci:.1f}%)" if mean_val > 1e-6
          else f"  {feat:15s}: {mean_val:.6f} (negligible)")

## ANALYSIS 4: Temporal importance stability

In [ ]:
print("\n" + "="*60)
print("TEMPORAL IMPORTANCE RANKING STABILITY")
print("="*60)

temp_mean = np.mean(boot_temp, axis=0)
temp_rank = np.argsort(temp_mean)[::-1]
print(f"Full temporal ranking (top 5): timesteps {temp_rank[:5]+1}")

# Check if top-3 timesteps are stable across bootstrap subsets
ranking_matches = 0
n_trials = 500
for trial in range(n_trials):
    idx = np.random.choice(n_boot, n_boot // 2, replace=False)
    sub_mean = np.mean(boot_temp[idx], axis=0)
    sub_rank = np.argsort(sub_mean)[::-1]
    if set(sub_rank[:3]) == set(temp_rank[:3]):
        ranking_matches += 1

print(f"Top-3 timestep stability (500 half-sample trials): {ranking_matches/n_trials*100:.1f}%")

## SAVE RESULTS

In [ ]:
pd.DataFrame(feat_ci).to_csv(tables_dir / 'shap_feature_ci_validation.csv', index=False)
pd.DataFrame(ci_results).to_csv(tables_dir / 'shap_ci_convergence.csv', index=False)

all_validation = {
    'feature_ci': feat_ci,
    'ci_convergence': ci_results,
    'top3_temporal_stability_pct': float(ranking_matches/n_trials*100),
    'full_feature_ranking': [features[i] for i in full_rank],
}

with open(tables_dir / 'shap_validation_results.json', 'w') as f:
    json.dump(all_validation, f, indent=2, default=str)

print("\n✅ SHAP validation complete")